In [1]:
import os
import re
import sys
import uuid
import chromadb

from dotenv import load_dotenv,find_dotenv
from unittest import loader
from lzma import CHECK_UNKNOWN
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_groq import ChatGroq


dotenv_path=find_dotenv()
load_dotenv(dotenv_path)
api_key=os.getenv("GROQ_API_KEY")

C:\Users\hp\AppData\Local\Temp\ipykernel_20604\3182380096.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader


In [2]:
def load_all_pdfs():
    folder_path = "data"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs


def split_docs(documents,chunk_size=1000,chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size= chunk_size,
        chunk_overlap = chunk_overlap
    )
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [3]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [4]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None
        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name, 
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )
        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")
            
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)
            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        # Add everything ONCE
        self.collection.add(
            ids=ids, 
            metadatas=all_metadata, 
            documents=documents_content, 
            embeddings=embeddings_list
        )
        print("total documents added in vector store =", len(documents_content))
        print("docs in collection:", self.collection.count())

In [5]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [6]:
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)]) 
    return response.content

In [7]:
all_pdf_documents = load_all_pdfs()
chunks = split_docs(all_pdf_documents)

total pdfs: 1
total pages: 16


In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embedding_manager = EmbeddingManager()
vector_store = VectorStoreManager()
texts = [doc.page_content for doc in chunks]
emebedding = embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks, emebedding)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\hp\AppData\Local\Temp\ipykernel_20604\4021223045.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


embedding dimensions= 384
initialized the vector store with collection: pdf_documents
docs in collection: 246


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

embeddings shape: (123, 384)
total documents added in vector store = 123
docs in collection: 369


In [10]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)
rag_retriever.retrieve("What is AI?")
llm = ChatGroq(groq_api_key= api_key, model="openai/gpt-oss-20b", temperature=0.1, max_tokens=1024)
answer = generate_output("What is AI?", rag_retriever, llm)
print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 0 documents


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 0 documents
we found no relevant context for the given query
Artificial Intelligence (AI) is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks include reasoning, learning, problem‑solving, perception, language understanding, and even creativity. 

Key aspects of AI:

| Aspect | What it means | Example |
|--------|---------------|---------|
| **Machine Learning** | Algorithms that improve from data | Image recognition, recommendation engines |
| **Neural Networks** | Models inspired by the brain’s structure | Deep learning for speech and vision |
| **Natural Language Processing** | Understanding and generating human language | Chatbots, translation services |
| **Robotics** | Physical agents that interact with the world | Autonomous drones, surgical robots |
| **Expert Systems** | Rule‑based decision making | Medical diagnosis assistants |

AI can be **narro